# LeetCode 1631 — Path With Minimum Effort bằng Python

Link đề: https://leetcode.com/problems/path-with-minimum-effort/ · Độ khó: **Medium** · Chủ đề: Array, Binary Search, Depth-First Search, Breadth-First Search, Union Find, Heap (Priority Queue), Matrix

Skeleton LeetCode:

```python
class Solution:
    def minimumEffortPath(self, heights: List[List[int]]) -> int:
        ...
```

## 1. Mục tiêu học

1. Mô hình hóa lưới thành **đồ thị**: mỗi ô là đỉnh, hai ô kề nhau có cạnh trọng số `|h₁ − h₂|`; bài toán là đường đi **minimax** (cực tiểu hóa cạnh lớn nhất trên đường) chứ không phải tổng.
2. Đi từ vét cạn đường đi → thử ngưỡng tuyến tính → **tìm kiếm nhị phân trên đáp án + BFS**.
3. Biến đổi **Dijkstra** cho phép kết hợp `max` thay vì `+` và hiểu vì sao vẫn đúng.
4. Dùng **Union-Find (Kruskal)**: thêm cạnh theo trọng số tăng dần tới khi góc trên-trái và góc dưới-phải liên thông.
5. Tránh các bẫy: chỉ đi phải/xuống, đánh dấu visited khi đẩy vào heap, biên tìm kiếm nhị phân, lưới một ô.

## 2. Đề bài & ví dụ

Bạn là người leo núi. Cho mảng 2 chiều `heights` kích thước `rows × columns`, `heights[row][col]` là độ cao ô `(row, col)`. Bạn đứng ở ô trên-trái `(0, 0)` và muốn tới ô dưới-phải `(rows−1, columns−1)`. Mỗi bước đi **lên, xuống, trái hoặc phải**.

**Effort** của một lộ trình là **giá trị lớn nhất** của chênh lệch độ cao tuyệt đối giữa hai ô liên tiếp trên lộ trình. Trả về effort **nhỏ nhất** cần để đi từ trên-trái tới dưới-phải.

**Ví dụ 1:** `heights = [[1,2,2],[3,8,2],[5,3,5]]` → `2`. Lộ trình `[1,3,5,3,5]` có chênh lệch lớn nhất 2, tốt hơn `[1,2,2,2,5]` (lớn nhất 3).

**Ví dụ 2:** `heights = [[1,2,3],[3,8,4],[5,3,5]]` → `1`. Lộ trình `[1,2,3,4,5]`.

**Ví dụ 3:** `heights = [[1,2,1,1,1],[1,2,1,2,1],[1,2,1,2,1],[1,2,1,2,1],[1,1,1,2,1]]` → `0`. Có lộ trình toàn số 1 (phải đi ngoằn ngoèo cả **lên** lẫn **xuống**).

**Ràng buộc:**
- `1 ≤ rows, columns ≤ 100`
- `1 ≤ heights[i][j] ≤ 10⁶`

In [1]:
import random
import time
import heapq
from collections import deque
from typing import List

examples = [
    ([[1, 2, 2], [3, 8, 2], [5, 3, 5]], 2),
    ([[1, 2, 3], [3, 8, 4], [5, 3, 5]], 1),
    ([[1, 2, 1, 1, 1], [1, 2, 1, 2, 1], [1, 2, 1, 2, 1], [1, 2, 1, 2, 1], [1, 1, 1, 2, 1]], 0),
]
for g, want in examples:
    print("\n".join(" ".join(f"{v:2}" for v in row) for row in g), f"\n-> kỳ vọng {want}\n")

 1  2  2
 3  8  2
 5  3  5 
-> kỳ vọng 2

 1  2  3
 3  8  4
 5  3  5 
-> kỳ vọng 1

 1  2  1  1  1
 1  2  1  2  1
 1  2  1  2  1
 1  2  1  2  1
 1  1  1  2  1 
-> kỳ vọng 0



## 3. Kiến thức nền

### 3.1 Duyệt 4 ô kề trong lưới

Dùng mảng hướng `DIRS = [(1,0),(-1,0),(0,1),(0,-1)]` và kiểm tra biên `0 ≤ r < m`, `0 ≤ c < n`.

In [2]:
DIRS = [(1, 0), (-1, 0), (0, 1), (0, -1)]

def neighbors(r, c, m, n):
    for dr, dc in DIRS:
        nr, nc = r + dr, c + dc
        if 0 <= nr < m and 0 <= nc < n:
            yield nr, nc

print(list(neighbors(0, 0, 3, 3)), list(neighbors(1, 1, 3, 3)))

[(1, 0), (0, 1)] [(2, 1), (0, 1), (1, 2), (1, 0)]


### 3.2 BFS với `deque` trên lưới

BFS từ `(0,0)` chỉ đi qua các cạnh thỏa điều kiện (ở đây: chênh lệch `≤ limit`) cho biết ô đích **có tới được không**. Mỗi ô thăm một lần ⇒ `O(m·n)`.

In [3]:
def reachable(heights, limit):
    m, n = len(heights), len(heights[0])
    seen = [[False] * n for _ in range(m)]
    seen[0][0] = True
    q = deque([(0, 0)])
    while q:
        r, c = q.popleft()
        if (r, c) == (m - 1, n - 1):
            return True
        for nr, nc in neighbors(r, c, m, n):
            if not seen[nr][nc] and abs(heights[nr][nc] - heights[r][c]) <= limit:
                seen[nr][nc] = True
                q.append((nr, nc))
    return False

g = examples[0][0]
print([(lim, reachable(g, lim)) for lim in range(0, 5)])

[(0, False), (1, False), (2, True), (3, True), (4, True)]


### 3.3 Hàng đợi ưu tiên `heapq`

`heapq.heappush(pq, (key, ...))` và `heapq.heappop(pq)` luôn lấy phần tử có `key` nhỏ nhất, mỗi thao tác `O(log size)`. Tuple so sánh theo thành phần đầu trước.

In [4]:
pq = []
for item in [(5, "e"), (1, "a"), (3, "c"), (1, "b")]:
    heapq.heappush(pq, item)
print([heapq.heappop(pq) for _ in range(len(pq))])

[(1, 'a'), (1, 'b'), (3, 'c'), (5, 'e')]


### 3.4 Union-Find (Disjoint Set Union)

Quản lý các nhóm đỉnh liên thông: `find(x)` trả đại diện nhóm (có **nén đường**), `union(a, b)` gộp hai nhóm (**theo kích thước**). Gần như `O(1)` mỗi thao tác.

In [5]:
class DSU:
    def __init__(self, n):
        self.p = list(range(n)); self.sz = [1] * n
    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]      # nén đường (halving)
            x = self.p[x]
        return x
    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a == b: return False
        if self.sz[a] < self.sz[b]: a, b = b, a
        self.p[b] = a; self.sz[a] += self.sz[b]
        return True

d = DSU(5); d.union(0, 1); d.union(3, 4)
print(d.find(1) == d.find(0), d.find(1) == d.find(3)); d.union(1, 4); print(d.find(0) == d.find(3))

True False
True


### 3.5 Tìm kiếm nhị phân trên **đáp án**

Nếu tính chất `P(x)` **đơn điệu** (sai…sai, đúng…đúng) thì tìm `x` nhỏ nhất thỏa `P` bằng `lo, hi` và `while lo < hi: mid = (lo+hi)//2; if P(mid): hi = mid else: lo = mid + 1`.

In [6]:
def first_true(lo, hi, P):
    while lo < hi:
        mid = (lo + hi) // 2
        if P(mid): hi = mid
        else: lo = mid + 1
    return lo

print(first_true(0, 100, lambda x: x * x >= 50))     # 8

8


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1 `[[1,2,2],[3,8,2],[5,3,5]]`. Viết trọng số cạnh (chênh lệch) giữa các ô kề:

```
 1 -1- 2 -0- 2
 |2    |6    |0
 3 -5- 8 -6- 2
 |2    |5    |3
 5 -2- 3 -2- 5
```

- Lộ trình đi theo cột trái rồi hàng dưới: `1→3→5→3→5` dùng các cạnh `2, 2, 2, 2` ⇒ effort `2`.
- Lộ trình `1→2→2→2→5` dùng `1, 0, 0, 3` ⇒ effort `3`.
- Không có đường nào dùng toàn cạnh `≤ 1`: từ `1` chỉ sang `2`, `2`, `2` (hàng trên + ô (1,2)) rồi bị chặn bởi cạnh `3` xuống `5`.

**Góc nhìn ngưỡng:** "effort ≤ `x` có khả thi?" ⇔ chỉ giữ cạnh có trọng số `≤ x`, hai góc có liên thông? Khi `x` tăng thì càng nhiều cạnh ⇒ **đơn điệu** ⇒ tìm `x` nhỏ nhất.

In [7]:
g = examples[0][0]
for lim in range(4):
    print(f"giới hạn {lim}: tới được đích? {reachable(g, lim)}")

giới hạn 0: tới được đích? False
giới hạn 1: tới được đích? False
giới hạn 2: tới được đích? True
giới hạn 3: tới được đích? True


## 5. Cách 1 — Brute Force / baseline: liệt kê mọi đường đi đơn

**Ý tưởng:** DFS quay lui thử mọi đường đi không lặp ô từ `(0,0)` tới đích, theo dõi cạnh lớn nhất hiện tại; lấy min. Cắt tỉa nhẹ: nếu cạnh lớn nhất hiện tại đã `≥` đáp án tốt nhất thì bỏ nhánh.

In [8]:
def effort_brute(heights):
    m, n = len(heights), len(heights[0])
    best = float("inf")
    seen = [[False] * n for _ in range(m)]
    def dfs(r, c, cur):
        nonlocal best
        if cur >= best:
            return
        if (r, c) == (m - 1, n - 1):
            best = cur; return
        seen[r][c] = True
        for nr, nc in neighbors(r, c, m, n):
            if not seen[nr][nc]:
                dfs(nr, nc, max(cur, abs(heights[nr][nc] - heights[r][c])))
        seen[r][c] = False
    dfs(0, 0, 0)
    return best

for g, want in examples:
    print(effort_brute(g), "kỳ vọng", want)

2 kỳ vọng 2
1 kỳ vọng 1
0 kỳ vọng 0


**Độ phức tạp:** số đường đi đơn trên lưới tăng **theo hàm mũ** (lưới `5×5` có hàng nghìn đường từ góc tới góc, `7×7` hàng trăm triệu). Không dùng được với `100×100`.

**Nút thắt:** ta thử lại vô số đường đi chỉ khác nhau chút ít. Thay vì hỏi "đường nào tốt nhất", hỏi câu **quyết định**: "có đường nào với effort `≤ x` không?" — câu này trả lời được bằng một BFS `O(m·n)`.

In [9]:
rnd = random.Random(0)
for k in (3, 4, 5):
    g = [[rnd.randint(1, 50) for _ in range(k)] for _ in range(k)]
    t = time.perf_counter(); effort_brute(g)
    print(f"{k}x{k}: {time.perf_counter() - t:.4f}s")

3x3: 0.0000s
4x4: 0.0000s
5x5: 0.0021s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — đơn điệu theo ngưỡng.** Đặt `P(x)` = "tồn tại đường đi chỉ dùng cạnh `≤ x`". Nếu `P(x)` đúng thì `P(x')` đúng với mọi `x' ≥ x`. Đáp án là `x` nhỏ nhất thỏa `P`, và nó **luôn là trọng số của một cạnh nào đó** (hoặc 0).
- Thử lần lượt mọi trọng số cạnh tăng dần ⇒ `O(E · mn)` (Cách 2).
- Tìm kiếm nhị phân trên `[0, 10⁶]` ⇒ `O(mn · log 10⁶)` (Cách 3).

**Insight 2 — Dijkstra với phép `max`.** Gọi `dist[v]` = effort nhỏ nhất để tới `v`. Đi tiếp qua cạnh `w` cho `max(dist[u], w)`. Hàm này **không giảm** theo đường đi (thêm cạnh không làm effort nhỏ đi) — đúng điều kiện để Dijkstra chọn tham lam ô có `dist` nhỏ nhất và "chốt" nó (Cách 4).

**Insight 3 — Kruskal.** Thêm cạnh theo trọng số tăng dần vào Union-Find; **cạnh làm `(0,0)` và đích liên thông lần đầu** chính là đáp án — vì ngay trước đó (mọi cạnh nhỏ hơn) chưa liên thông, ngay sau đó thì đã có đường dùng toàn cạnh `≤` cạnh vừa thêm (Cách 5). Đây cũng là tính chất của **cây khung nhỏ nhất**: đường minimax giữa hai đỉnh nằm trên MST.

Quy tắc "nếu… thì…":
- **Nếu** bài hỏi "cực tiểu hóa **giá trị lớn nhất**" **thì** thử tìm kiếm nhị phân trên đáp án hoặc Dijkstra/Kruskal biến thể minimax.
- **Nếu** lưới cho đi **4 hướng** **thì** không dùng DP phải/xuống.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Thử mọi ngưỡng theo thứ tự tăng dần + BFS

Lấy tập trọng số cạnh phân biệt (thêm 0), sắp tăng dần, BFS với từng ngưỡng cho tới khi tới được đích. Đúng nhưng chậm khi có nhiều trọng số khác nhau.

In [10]:
def edge_weights(heights):
    m, n = len(heights), len(heights[0])
    ws = {0}
    for r in range(m):
        for c in range(n):
            if r + 1 < m: ws.add(abs(heights[r + 1][c] - heights[r][c]))
            if c + 1 < n: ws.add(abs(heights[r][c + 1] - heights[r][c]))
    return sorted(ws)

def effort_linear(heights):
    for w in edge_weights(heights):
        if reachable(heights, w):
            return w

for g, want in examples:
    print(effort_linear(g), "kỳ vọng", want, "| số ngưỡng ứng viên:", len(edge_weights(g)))

2 kỳ vọng 2 | số ngưỡng ứng viên: 6
1 kỳ vọng 1 | số ngưỡng ứng viên: 6
0 kỳ vọng 0 | số ngưỡng ứng viên: 2


**Độ phức tạp:** Time `O(K · m·n)` với `K` = số trọng số phân biệt (tới `2mn ≈ 2·10⁴`) ⇒ worst `2·10⁸` — quá chậm cho Python. Space `O(m·n)`.

### 7.2 Cách 3 — Tìm kiếm nhị phân trên đáp án + BFS

Thay quét tuyến tính bằng tìm kiếm nhị phân: trên khoảng `[0, 10⁶]` (hoặc trên danh sách trọng số đã sắp — chỉ `~15` lần BFS).

In [11]:
def effort_binary(heights):
    lo, hi = 0, 10**6
    while lo < hi:
        mid = (lo + hi) // 2
        if reachable(heights, mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

def effort_binary_on_weights(heights):
    ws = edge_weights(heights)
    lo, hi = 0, len(ws) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if reachable(heights, ws[mid]): hi = mid
        else: lo = mid + 1
    return ws[lo]

for g, want in examples:
    print(effort_binary(g), effort_binary_on_weights(g), "kỳ vọng", want)

2 2 kỳ vọng 2
1 1 kỳ vọng 1
0 0 kỳ vọng 0


Code nộp LeetCode của Cách 3:

```python
class Solution:
    def minimumEffortPath(self, heights: List[List[int]]) -> int:
        m, n = len(heights), len(heights[0])
        def ok(limit):
            seen = [[False] * n for _ in range(m)]
            seen[0][0] = True
            q = deque([(0, 0)])
            while q:
                r, c = q.popleft()
                if r == m - 1 and c == n - 1:
                    return True
                for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < m and 0 <= nc < n and not seen[nr][nc] \
                            and abs(heights[nr][nc] - heights[r][c]) <= limit:
                        seen[nr][nc] = True
                        q.append((nr, nc))
            return False
        lo, hi = 0, 10**6
        while lo < hi:
            mid = (lo + hi) // 2
            if ok(mid): hi = mid
            else: lo = mid + 1
        return lo
```

**Độ phức tạp:** Time `O(m·n · log H)` với `H = 10⁶` (khoảng 20 lần BFS), Space `O(m·n)`.

### 7.3 Cách 4 — Dijkstra biến thể minimax

Từng đoạn:
1. `dist[0][0] = 0`, các ô khác `∞`; heap `[(0, 0, 0)]`.
2. Lấy ô có effort nhỏ nhất. Nếu là **bản cũ** (`e > dist`) bỏ qua. Nếu là đích ⇒ trả `e` (đã chốt).
3. Với mỗi ô kề: `ne = max(e, |Δh|)`; nếu `ne < dist[kề]` thì cập nhật và đẩy vào heap.

In [12]:
def effort_dijkstra(heights):
    m, n = len(heights), len(heights[0])
    INF = float("inf")
    dist = [[INF] * n for _ in range(m)]
    dist[0][0] = 0
    pq = [(0, 0, 0)]
    while pq:
        e, r, c = heapq.heappop(pq)
        if e > dist[r][c]:
            continue                                   # bản cũ, đã có đường tốt hơn
        if r == m - 1 and c == n - 1:
            return e
        for nr, nc in neighbors(r, c, m, n):
            ne = max(e, abs(heights[nr][nc] - heights[r][c]))
            if ne < dist[nr][nc]:
                dist[nr][nc] = ne
                heapq.heappush(pq, (ne, nr, nc))
    return 0

for g, want in examples:
    print(effort_dijkstra(g), "kỳ vọng", want)

2 kỳ vọng 2
1 kỳ vọng 1
0 kỳ vọng 0


Code nộp LeetCode của Cách 4: xem mục 9.

**Độ phức tạp:** Time `O(m·n · log(m·n))` (mỗi ô có ≤ 4 cạnh, mỗi lần đẩy heap `O(log)`), Space `O(m·n)`.

### 7.4 Cách 5 — Kruskal / Union-Find

Từng đoạn:
1. Đánh số ô `id = r·n + c`. Tạo danh sách cạnh `(w, id1, id2)` cho cặp kề phải và kề dưới.
2. Sắp cạnh tăng dần theo `w`.
3. Lần lượt `union`; ngay khi `find(0) == find(mn − 1)` ⇒ trả `w`.
4. Lưới một ô: không có cạnh ⇒ trả `0`.

In [13]:
def effort_kruskal(heights):
    m, n = len(heights), len(heights[0])
    if m * n == 1:
        return 0
    edges = []
    for r in range(m):
        for c in range(n):
            i = r * n + c
            if r + 1 < m: edges.append((abs(heights[r + 1][c] - heights[r][c]), i, i + n))
            if c + 1 < n: edges.append((abs(heights[r][c + 1] - heights[r][c]), i, i + 1))
    edges.sort()
    dsu = DSU(m * n)
    target = m * n - 1
    for w, a, b in edges:
        dsu.union(a, b)
        if dsu.find(0) == dsu.find(target):
            return w
    return 0

for g, want in examples:
    print(effort_kruskal(g), "kỳ vọng", want)

2 kỳ vọng 2
1 kỳ vọng 1
0 kỳ vọng 0


Code nộp LeetCode của Cách 5:

```python
class Solution:
    def minimumEffortPath(self, heights: List[List[int]]) -> int:
        m, n = len(heights), len(heights[0])
        if m * n == 1:
            return 0
        edges = []
        for r in range(m):
            for c in range(n):
                i = r * n + c
                if r + 1 < m: edges.append((abs(heights[r+1][c] - heights[r][c]), i, i + n))
                if c + 1 < n: edges.append((abs(heights[r][c+1] - heights[r][c]), i, i + 1))
        edges.sort()
        parent = list(range(m * n))
        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x
        for w, a, b in edges:
            parent[find(a)] = find(b)
            if find(0) == find(m * n - 1):
                return w
        return 0
```

**Độ phức tạp:** Time `O(E log E)` với `E ≈ 2mn` (chủ yếu là sắp xếp), Space `O(m·n)`.

## 8. TRACE từng bước

Trace **Dijkstra** trên ví dụ 2 `[[1,2,3],[3,8,4],[5,3,5]]`:

| pop (effort, ô) | cập nhật |
|---|---|
| (0, (0,0)) | (1,0)←2, (0,1)←1 |
| (1, (0,1)) | (1,1)←6, (0,2)←1 |
| (1, (0,2)) | (1,2)←1 |
| (1, (1,2)) | (2,2)←1, (1,1)←4 |
| (1, (2,2)) | đích ⇒ trả 1 |

In [14]:
def trace_dijkstra(heights):
    m, n = len(heights), len(heights[0])
    INF = float("inf")
    dist = [[INF] * n for _ in range(m)]; dist[0][0] = 0
    pq = [(0, 0, 0)]
    while pq:
        e, r, c = heapq.heappop(pq)
        if e > dist[r][c]:
            print(f"pop ({e}, ({r},{c})) -> bản cũ, bỏ qua"); continue
        if (r, c) == (m - 1, n - 1):
            print(f"pop ({e}, ({r},{c})) -> ĐÍCH, đáp án {e}"); return e
        ups = []
        for nr, nc in neighbors(r, c, m, n):
            ne = max(e, abs(heights[nr][nc] - heights[r][c]))
            if ne < dist[nr][nc]:
                dist[nr][nc] = ne; heapq.heappush(pq, (ne, nr, nc)); ups.append(f"({nr},{nc})←{ne}")
        print(f"pop ({e}, ({r},{c})) -> cập nhật: {', '.join(ups) or '-'}")

assert trace_dijkstra(examples[1][0]) == 1

pop (0, (0,0)) -> cập nhật: (1,0)←2, (0,1)←1
pop (1, (0,1)) -> cập nhật: (1,1)←6, (0,2)←1
pop (1, (0,2)) -> cập nhật: (1,2)←1
pop (1, (1,2)) -> cập nhật: (2,2)←1, (1,1)←4
pop (1, (2,2)) -> ĐÍCH, đáp án 1


In [15]:
def trace_binary(heights):
    lo, hi = 0, 10**6
    while lo < hi:
        mid = (lo + hi) // 2
        ok = reachable(heights, mid)
        print(f"lo={lo:>7} hi={hi:>7} mid={mid:>7} -> {'tới được, hi=mid' if ok else 'không, lo=mid+1'}")
        if ok: hi = mid
        else: lo = mid + 1
    print("đáp án:", lo)
    return lo

assert trace_binary(examples[0][0]) == 2

lo=      0 hi=1000000 mid= 500000 -> tới được, hi=mid
lo=      0 hi= 500000 mid= 250000 -> tới được, hi=mid
lo=      0 hi= 250000 mid= 125000 -> tới được, hi=mid
lo=      0 hi= 125000 mid=  62500 -> tới được, hi=mid
lo=      0 hi=  62500 mid=  31250 -> tới được, hi=mid
lo=      0 hi=  31250 mid=  15625 -> tới được, hi=mid
lo=      0 hi=  15625 mid=   7812 -> tới được, hi=mid
lo=      0 hi=   7812 mid=   3906 -> tới được, hi=mid
lo=      0 hi=   3906 mid=   1953 -> tới được, hi=mid
lo=      0 hi=   1953 mid=    976 -> tới được, hi=mid
lo=      0 hi=    976 mid=    488 -> tới được, hi=mid
lo=      0 hi=    488 mid=    244 -> tới được, hi=mid
lo=      0 hi=    244 mid=    122 -> tới được, hi=mid
lo=      0 hi=    122 mid=     61 -> tới được, hi=mid
lo=      0 hi=     61 mid=     30 -> tới được, hi=mid
lo=      0 hi=     30 mid=     15 -> tới được, hi=mid
lo=      0 hi=     15 mid=      7 -> tới được, hi=mid
lo=      0 hi=      7 mid=      3 -> tới được, hi=mid
lo=      0 hi=      3 mid=  

In [16]:
def trace_kruskal(heights):
    m, n = len(heights), len(heights[0])
    edges = []
    for r in range(m):
        for c in range(n):
            if r + 1 < m: edges.append((abs(heights[r + 1][c] - heights[r][c]), (r, c), (r + 1, c)))
            if c + 1 < n: edges.append((abs(heights[r][c + 1] - heights[r][c]), (r, c), (r, c + 1)))
    edges.sort()
    dsu = DSU(m * n)
    for w, a, b in edges:
        merged = dsu.union(a[0] * n + a[1], b[0] * n + b[1])
        done = dsu.find(0) == dsu.find(m * n - 1)
        print(f"cạnh w={w}: {a}-{b} {'gộp' if merged else 'đã cùng nhóm'}{'  => (0,0) và đích liên thông!' if done else ''}")
        if done:
            return w

assert trace_kruskal(examples[0][0]) == 2

cạnh w=0: (0, 1)-(0, 2) gộp
cạnh w=0: (0, 2)-(1, 2) gộp
cạnh w=1: (0, 0)-(0, 1) gộp
cạnh w=2: (0, 0)-(1, 0) gộp
cạnh w=2: (1, 0)-(2, 0) gộp
cạnh w=2: (2, 0)-(2, 1) gộp
cạnh w=2: (2, 1)-(2, 2) gộp  => (0,0) và đích liên thông!


## 9. Code nộp LeetCode

Chọn Dijkstra minimax (Cách 4) — một lần duyệt, không phụ thuộc miền giá trị độ cao.

In [17]:
class Solution:
    def minimumEffortPath(self, heights: List[List[int]]) -> int:
        m, n = len(heights), len(heights[0])
        dist = [[float("inf")] * n for _ in range(m)]
        dist[0][0] = 0
        pq = [(0, 0, 0)]
        while pq:
            e, r, c = heapq.heappop(pq)
            if e > dist[r][c]:
                continue
            if r == m - 1 and c == n - 1:
                return e
            for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                nr, nc = r + dr, c + dc
                if 0 <= nr < m and 0 <= nc < n:
                    ne = max(e, abs(heights[nr][nc] - heights[r][c]))
                    if ne < dist[nr][nc]:
                        dist[nr][nc] = ne
                        heapq.heappush(pq, (ne, nr, nc))
        return 0

for g, want in examples:
    got = Solution().minimumEffortPath(g)
    print(got, "OK" if got == want else "SAI")

2 OK
1 OK
0 OK


## 10. Vì sao thuật toán đúng

**Tìm kiếm nhị phân (Cách 3).** `P(x)` đơn điệu (thêm cạnh không làm mất đường). Đáp án `x*` thỏa `P(x*)` và `¬P(x* − 1)`. Vòng lặp `lo < hi` giữ invariant `P(hi)` đúng và `P(lo − 1)` sai (`P(10⁶)` luôn đúng vì mọi chênh lệch `< 10⁶`), nên dừng tại `lo = x*`.

**Dijkstra minimax (Cách 4).** Định nghĩa chi phí đường đi `cost(p) = max cạnh`. Tính chất cần cho Dijkstra: nối thêm cạnh không làm giảm chi phí (`max(e, w) ≥ e`). Chứng minh chuẩn: khi ô `u` được lấy ra lần đầu với `e = dist[u]`, giả sử tồn tại đường tốt hơn `cost < e`; đường đó rời tập đã chốt tại một ô `y` nào đó có `dist[y] ≤ cost < e` ⇒ `y` lẽ ra phải được lấy ra trước `u` — mâu thuẫn. Vậy `dist[u]` tối ưu, và đích được trả về đúng lúc được chốt.

**Kruskal (Cách 5).** Gọi `w*` là trọng số cạnh làm hai góc liên thông lần đầu. Trước khi thêm nó, đồ thị gồm mọi cạnh `< w*` (và một số cạnh `= w*`) chưa nối hai góc ⇒ không có đường dùng toàn cạnh `< w*` ⇒ đáp án `≥ w*`. Sau khi thêm, có đường dùng toàn cạnh `≤ w*` ⇒ đáp án `≤ w*`.

## 11. Phân tích Time / Space Complexity

Gọi `V = m·n ≤ 10⁴` ô, `E ≈ 2V` cạnh, `H = 10⁶` độ cao lớn nhất.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn đường đi | hàm mũ | `O(V)` |
| 2. Thử mọi ngưỡng | `O(E · V)` | `O(V)` |
| 3. Nhị phân + BFS | `O(V log H)` | `O(V)` |
| 4. Dijkstra minimax | `O(E log V)` | `O(V)` |
| 5. Kruskal + DSU | `O(E log E)` | `O(E)` |

Dijkstra có thể dừng sớm khi chốt đích (average nhanh hơn worst). Tìm kiếm nhị phân luôn chạy đủ `~20` lần BFS.

In [18]:
rnd = random.Random(31)
big = [[rnd.randint(1, 10**6) for _ in range(100)] for _ in range(100)]
smooth = [[(r * 7 + c * 3) % 50 + 1 for c in range(100)] for r in range(100)]
for f in (effort_binary, effort_dijkstra, effort_kruskal):
    for g, name in ((big, "ngẫu nhiên"), (smooth, "trơn")):
        t = time.perf_counter(); v = f(g)
        print(f"{f.__name__:16s} {name:10s} 100x100 -> {v:>7} trong {time.perf_counter() - t:.3f}s")

effort_binary    ngẫu nhiên 100x100 ->  479514 trong 0.069s


effort_binary    trơn       100x100 ->      43 trong 0.096s


effort_dijkstra  ngẫu nhiên 100x100 ->  479514 trong 0.013s
effort_dijkstra  trơn       100x100 ->      43 trong 0.009s
effort_kruskal   ngẫu nhiên 100x100 ->  479514 trong 0.013s


effort_kruskal   trơn       100x100 ->      43 trong 0.010s


## 12. Bộ test

- Ví dụ đề.
- Edge: lưới một ô `[[5]]` → 0; một hàng, một cột (đường duy nhất ⇒ đáp án là cạnh lớn nhất); lưới phẳng → 0; lưới cần đi **lên/trái** (ví dụ 3).
- Ngẫu nhiên seed cố định: lưới nhỏ (≤ 4×4, ≤ 3×5) so với vét cạn; lưới lớn đúng ràng buộc so các cách với nhau.

In [19]:
ALL = [effort_linear, effort_binary, effort_binary_on_weights, effort_dijkstra, effort_kruskal,
       lambda g: Solution().minimumEffortPath(g)]
fixed = examples + [
    ([[5]], 0),
    ([[1, 10, 4, 6]], 9),
    ([[1], [3], [8], [2]], 6),
    ([[7, 7], [7, 7]], 0),
    ([[1, 1000000]], 999999),
]
for g, want in fixed:
    assert effort_brute(g) == want, g
    for f in ALL:
        assert f(g) == want, (f, g)
print("ví dụ cố định: OK")

ví dụ cố định: OK


In [20]:
rnd = random.Random(1631)
for _ in range(300):
    m, n = rnd.choice([(1, 5), (2, 2), (2, 3), (3, 3), (3, 4), (4, 4), (3, 5), (5, 1)])
    hi_val = rnd.choice([3, 10, 10**6])
    g = [[rnd.randint(1, hi_val) for _ in range(n)] for _ in range(m)]
    want = effort_brute(g)
    for f in ALL:
        assert f(g) == want, (f, g)
for _ in range(6):
    m, n = rnd.randint(1, 100), rnd.randint(1, 100)
    g = [[rnd.randint(1, rnd.choice([5, 10**6])) for _ in range(n)] for _ in range(m)]
    assert len({f(g) for f in ALL[1:]}) == 1
print("random tests: OK")

random tests: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — DP chỉ đi phải / xuống

In [21]:
def wrong_right_down(heights):
    m, n = len(heights), len(heights[0])
    INF = float("inf")
    f = [[INF] * n for _ in range(m)]; f[0][0] = 0
    for r in range(m):
        for c in range(n):
            if r: f[r][c] = min(f[r][c], max(f[r - 1][c], abs(heights[r][c] - heights[r - 1][c])))
            if c: f[r][c] = min(f[r][c], max(f[r][c - 1], abs(heights[r][c] - heights[r][c - 1])))
    return f[m - 1][n - 1]

g3 = examples[2][0]
print("sai:", wrong_right_down(g3), "| đúng:", effort_dijkstra(g3))

sai: 1 | đúng: 0


**Vì sao sai:** đề cho đi **4 hướng**. Ở ví dụ 3, đường toàn số 1 phải đi xuống cột 0, sang phải, **đi lên** cột 2, sang phải rồi đi xuống — DP phải/xuống không biểu diễn được. **Sửa:** dùng thuật toán đồ thị (Dijkstra/BFS/DSU).

### Lỗi 2 — Dùng Dijkstra "tổng" thay vì "max"

In [22]:
def wrong_sum(heights):
    m, n = len(heights), len(heights[0])
    dist = [[float("inf")] * n for _ in range(m)]; dist[0][0] = 0
    pq = [(0, 0, 0)]
    while pq:
        e, r, c = heapq.heappop(pq)
        if e > dist[r][c]: continue
        for nr, nc in neighbors(r, c, m, n):
            ne = e + abs(heights[nr][nc] - heights[r][c])          # SAI: cộng dồn
            if ne < dist[nr][nc]:
                dist[nr][nc] = ne; heapq.heappush(pq, (ne, nr, nc))
    return dist[m - 1][n - 1]

g1 = examples[0][0]
print("sai:", wrong_sum(g1), "| đúng:", effort_dijkstra(g1))

sai: 4 | đúng: 2


**Vì sao sai:** effort là **cạnh lớn nhất**, không phải **tổng**. Đường ngắn nhất theo tổng (ở đây `1→2→2→2→5`, tổng 4) có cạnh lớn nhất 3, trong khi đường `1→3→5→3→5` (tổng 8) chỉ có cạnh lớn nhất 2. **Sửa:** `ne = max(e, w)`.

### Lỗi 3 — Đánh dấu visited ngay khi **đẩy** vào heap

In [23]:
def wrong_mark_on_push(heights):
    m, n = len(heights), len(heights[0])
    seen = {(0, 0)}; pq = [(0, 0, 0)]
    while pq:
        e, r, c = heapq.heappop(pq)
        if (r, c) == (m - 1, n - 1): return e
        for nr, nc in neighbors(r, c, m, n):
            if (nr, nc) not in seen:
                seen.add((nr, nc))                                  # SAI: chốt quá sớm
                heapq.heappush(pq, (max(e, abs(heights[nr][nc] - heights[r][c])), nr, nc))
    return 0

cx = [[9, 6, 3], [6, 5, 9]]
print("sai:", wrong_mark_on_push(cx), "| đúng:", effort_brute(cx))

sai: 6 | đúng: 4


**Vì sao sai:** lần đầu một ô được **nhìn thấy** chưa chắc là qua đường tốt nhất; khóa nó lại khiến đường tốt hơn tới sau bị bỏ. Ở đây ô `(1,2)` (độ cao 9) bị khóa với effort 6 (từ `3`), trong khi đi vòng qua `5` chỉ cần 4. **Sửa:** dùng `dist` và cho phép cập nhật khi tìm được giá trị nhỏ hơn; chỉ "chốt" khi **lấy ra** (pop).

### Lỗi 4 — Biên tìm kiếm nhị phân sai (`hi = mid − 1` khi `P(mid)` đúng)

In [24]:
def wrong_binary(heights):
    lo, hi = 0, 10**6
    while lo < hi:
        mid = (lo + hi) // 2
        if reachable(heights, mid):
            hi = mid - 1                                            # SAI: có thể bỏ qua chính đáp án
        else:
            lo = mid + 1
    return lo

for g, want in examples:
    print("sai:", wrong_binary(g), "| đúng:", want)

sai: 1 | đúng: 2
sai: 1 | đúng: 1
sai: 0 | đúng: 0


**Vì sao sai:** khi `P(mid)` đúng, `mid` **có thể chính là** đáp án nhỏ nhất; `hi = mid − 1` loại nó đi và vòng lặp có thể kết thúc tại giá trị mà `P` sai. **Sửa:** mẫu "first true": `hi = mid` khi đúng, `lo = mid + 1` khi sai, điều kiện `lo < hi`.

### Lỗi 5 — Kruskal quên lưới một ô (hoặc trả mặc định sai)

In [25]:
def wrong_kruskal_single(heights):
    m, n = len(heights), len(heights[0])
    edges = []
    for r in range(m):
        for c in range(n):
            i = r * n + c
            if r + 1 < m: edges.append((abs(heights[r + 1][c] - heights[r][c]), i, i + n))
            if c + 1 < n: edges.append((abs(heights[r][c + 1] - heights[r][c]), i, i + 1))
    edges.sort()
    dsu = DSU(m * n)
    for w, a, b in edges:
        dsu.union(a, b)
        if dsu.find(0) == dsu.find(m * n - 1):
            return w
    return -1                                                       # SAI: lưới 1 ô không có cạnh

print("sai:", wrong_kruskal_single([[5]]), "| đúng:", effort_dijkstra([[5]]))

sai: -1 | đúng: 0


**Vì sao sai:** `[[5]]` — xuất phát đã ở đích, effort 0, nhưng không có cạnh nào để vòng lặp trả về. **Sửa:** kiểm tra `m·n == 1` trước (hoặc kiểm tra liên thông **trước** khi duyệt cạnh).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Minimize the maximum**" / "**maximize the minimum**" trên đường đi ⇒ ba lựa chọn chuẩn: **nhị phân đáp án + BFS/DFS**, **Dijkstra với `max`/`min`**, **Kruskal/DSU**.
- Lưới 4 hướng, không có thứ tự tự nhiên ⇒ **không** dùng DP phải/xuống.
- Ràng buộc `100×100`, giá trị `10⁶` ⇒ `O(V log V)` hay `O(V log H)` đều ổn.
- Dijkstra: luôn có dòng `if e > dist[r][c]: continue` (lazy deletion).
- Đáp án luôn là **trọng số của một cạnh** (hoặc 0) ⇒ có thể nhị phân trên danh sách trọng số đã sắp.

## 15. Biến thể của bài

1. **LC 778 Swim in Rising Water** (Hard) — minimax trên **giá trị ô** thay vì chênh lệch.
2. **LC 1102 Path With Maximum Minimum Value** — **maximin**: Dijkstra với max-heap và `min`.
3. **LC 1514 Path with Maximum Probability** — Dijkstra với phép nhân xác suất.
4. **LC 1970 Last Day Where You Can Still Cross** — nhị phân đáp án hoặc DSU ngược thời gian.
5. **LC 2812 Find the Safest Path in a Grid** — BFS đa nguồn + maximin.
6. **Bottleneck path / minimax path trên đồ thị tổng quát** — đường đi trên MST.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | mọi đường đi đơn | mũ | `O(V)` | oracle lưới ≤ 4×4 |
| 2. Thử mọi ngưỡng | BFS cho từng trọng số tăng dần | `O(EV)` | `O(V)` | bước đệm để thấy tính đơn điệu |
| 3. Nhị phân + BFS | đơn điệu ⇒ nhị phân | `O(V log H)` | `O(V)` | dễ viết, dễ chứng minh |
| 4. Dijkstra minimax | đổi `+` thành `max` | `O(E log V)` | `O(V)` | lời giải chuẩn, dừng sớm |
| 5. Kruskal + DSU | thêm cạnh tăng dần tới khi liên thông | `O(E log E)` | `O(E)` | khi quen DSU; nhiều truy vấn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `[[1]]`, `[[1,10]]`, `[[3,3],[3,3]]`, `[[1,2],[5,3]]`.

**Bài 2 (tính tay):** chạy tay Kruskal trên ví dụ 2: liệt kê các cạnh trọng số 0 và 1, chỉ ra thời điểm hai góc liên thông.

**Bài 3 (điền TODO):** viết `can_reach(heights, limit)` bằng **DFS dùng stack** (không đệ quy).

**Bài 4 (tự viết lại):** viết lại Cách 5 (Kruskal) không xem code, dùng DSU có union-by-size.

**Bài 5 (mini challenge):** viết `best_path(heights)` trả về **danh sách ô** của một lộ trình đạt effort nhỏ nhất (Dijkstra + mảng `parent`). Kiểm tra lộ trình hợp lệ và effort của nó bằng đáp án.

In [26]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def can_reach(heights, limit):
    m, n = len(heights), len(heights[0])
    stack = [(0, 0)]
    seen = {(0, 0)}
    # TODO: pop, nếu là đích -> True; đẩy các ô kề chưa thấy có |Δh| <= limit
    return False

In [27]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def kruskal_again(heights):
    # TODO
    pass

def best_path(heights):
    # TODO: Dijkstra + parent[(r, c)] = ô trước đó
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: lưới 1 ô ⇒ 0; một hàng ⇒ đường duy nhất. Mức 2: lưới phẳng ⇒ 0. Mức 3: `0, 9, 0, 1` (`[[1,2],[5,3]]`: đi `1→2→3` effort 1).

**Bài 2.** Mức 1: cạnh trọng số 0: không có; trọng số 1: `(0,0)-(0,1)`, `(0,1)-(0,2)`, `(0,2)-(1,2)`, `(1,2)-(2,2)`. Mức 2: thêm 4 cạnh này theo thứ tự đã sắp. Mức 3: sau khi thêm `(1,2)-(2,2)`, `(0,0)` và `(2,2)` cùng nhóm ⇒ đáp án 1.

**Bài 3.** Mức 1: giống BFS nhưng `stack.pop()`. Mức 2: đánh dấu `seen` khi đẩy (với bài **quyết định** thì đúng — khác với Dijkstra!). Mức 3: trả `True` ngay khi pop ra đích.

**Bài 4.** Mức 1: id ô `r·n + c`. Mức 2: chỉ cần cạnh phải và cạnh dưới. Mức 3: nhớ trường hợp 1 ô.

**Bài 5.** Mức 1: khi cập nhật `dist[kề]` thì đặt `parent[kề] = (r, c)`. Mức 2: đi ngược từ đích về `(0,0)` rồi đảo. Mức 3: effort của lộ trình = `max |Δh|` các cặp liên tiếp, phải bằng đáp án.

## 19. Đáp án tham khảo cho bài tập

In [28]:
# Bài 1, 2
b1 = [Solution().minimumEffortPath(g) for g in ([[1]], [[1, 10]], [[3, 3], [3, 3]], [[1, 2], [5, 3]])]
print("Bài 1:", b1)
assert b1 == [0, 9, 0, 1]
assert trace_kruskal(examples[1][0]) == 1

Bài 1: [0, 9, 0, 1]
cạnh w=1: (0, 0)-(0, 1) gộp
cạnh w=1: (0, 1)-(0, 2) gộp
cạnh w=1: (0, 2)-(1, 2) gộp
cạnh w=1: (1, 2)-(2, 2) gộp  => (0,0) và đích liên thông!


In [29]:
# Bài 3
def can_reach_sol(heights, limit):
    m, n = len(heights), len(heights[0])
    stack, seen = [(0, 0)], {(0, 0)}
    while stack:
        r, c = stack.pop()
        if (r, c) == (m - 1, n - 1):
            return True
        for nr, nc in neighbors(r, c, m, n):
            if (nr, nc) not in seen and abs(heights[nr][nc] - heights[r][c]) <= limit:
                seen.add((nr, nc)); stack.append((nr, nc))
    return False

rnd = random.Random(3)
for _ in range(300):
    m, n = rnd.randint(1, 6), rnd.randint(1, 6)
    g = [[rnd.randint(1, 20) for _ in range(n)] for _ in range(m)]
    lim = rnd.randint(0, 20)
    assert can_reach_sol(g, lim) == reachable(g, lim)
print("Bài 3: OK")

Bài 3: OK


In [30]:
# Bài 4
def kruskal_again_sol(heights):
    m, n = len(heights), len(heights[0])
    if m * n == 1:
        return 0
    edges = sorted(
        [(abs(heights[r][c] - heights[r + 1][c]), r * n + c, (r + 1) * n + c) for r in range(m - 1) for c in range(n)] +
        [(abs(heights[r][c] - heights[r][c + 1]), r * n + c, r * n + c + 1) for r in range(m) for c in range(n - 1)])
    dsu = DSU(m * n)
    for w, a, b in edges:
        dsu.union(a, b)
        if dsu.find(0) == dsu.find(m * n - 1):
            return w

rnd = random.Random(4)
for _ in range(300):
    m, n = rnd.randint(1, 7), rnd.randint(1, 7)
    g = [[rnd.randint(1, 30) for _ in range(n)] for _ in range(m)]
    assert kruskal_again_sol(g) == effort_dijkstra(g)
print("Bài 4: OK")

Bài 4: OK


In [31]:
# Bài 5
def best_path_sol(heights):
    m, n = len(heights), len(heights[0])
    dist = [[float("inf")] * n for _ in range(m)]; dist[0][0] = 0
    parent = {(0, 0): None}
    pq = [(0, 0, 0)]
    while pq:
        e, r, c = heapq.heappop(pq)
        if e > dist[r][c]: continue
        if (r, c) == (m - 1, n - 1): break
        for nr, nc in neighbors(r, c, m, n):
            ne = max(e, abs(heights[nr][nc] - heights[r][c]))
            if ne < dist[nr][nc]:
                dist[nr][nc] = ne; parent[(nr, nc)] = (r, c)
                heapq.heappush(pq, (ne, nr, nc))
    path, cur = [], (m - 1, n - 1)
    while cur is not None:
        path.append(cur); cur = parent[cur]
    return path[::-1]

def path_effort(heights, path):
    return max((abs(heights[a][b] - heights[c][d]) for (a, b), (c, d) in zip(path, path[1:])), default=0)

p = best_path_sol(examples[0][0])
print("Bài 5 ví dụ 1:", p, "độ cao:", [examples[0][0][r][c] for r, c in p])
rnd = random.Random(5)
for _ in range(300):
    m, n = rnd.randint(1, 6), rnd.randint(1, 6)
    g = [[rnd.randint(1, 15) for _ in range(n)] for _ in range(m)]
    p = best_path_sol(g)
    assert p[0] == (0, 0) and p[-1] == (m - 1, n - 1)
    assert all(abs(a - c) + abs(b - d) == 1 for (a, b), (c, d) in zip(p, p[1:]))
    assert path_effort(g, p) == effort_dijkstra(g)
print("Bài 5: OK")

Bài 5 ví dụ 1: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2)] độ cao: [1, 3, 5, 3, 5]
Bài 5: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template Dijkstra minimax trên lưới:**

```python
dist = [[inf] * n for _ in range(m)]; dist[0][0] = 0
pq = [(0, 0, 0)]
while pq:
    e, r, c = heappop(pq)
    if e > dist[r][c]: continue            # bản cũ
    if (r, c) == target: return e           # chốt đích
    for nr, nc in 4 hướng trong biên:
        ne = max(e, abs(h[nr][nc] - h[r][c]))
        if ne < dist[nr][nc]:
            dist[nr][nc] = ne; heappush(pq, (ne, nr, nc))
```

**Template nhị phân đáp án:** `lo, hi = 0, MAX; while lo < hi: mid = (lo+hi)//2; hi = mid if ok(mid) else lo = mid+1`.

**Checklist:**
- [ ] Đi **4 hướng**, kiểm tra biên đúng?
- [ ] Kết hợp bằng **`max`**, không phải `+`?
- [ ] Dijkstra: chỉ chốt khi **pop**, có `if e > dist: continue`?
- [ ] Nhị phân: `hi = mid` khi đúng, `lo = mid + 1` khi sai?
- [ ] Lưới **1 ô** trả 0? Một hàng / một cột?